In [ ]:
import os, sys
from pathlib import Path

def _detect_platform():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        pass
    if Path('/workspace').exists() and os.environ.get('VAST_CONTAINERLABEL'):
        return 'vastai'
    if sys.platform == 'win32': return 'windows'
    if sys.platform == 'darwin': return 'mac'
    return None

PLATFORM = _detect_platform()

if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

PROJECT_ROOT = (
    Path('/content/drive/MyDrive/Thesis_Final/fake-news-detection')
    if PLATFORM == 'colab'
    else _nb_path.parents[1]
)
sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {'colab': '.env.colab', 'vastai': '.env.vastai', 'windows': '.env.windows', 'mac': '.env.mac'}
_env_file = PROJECT_ROOT / _env_map.get(PLATFORM, '.env')
if not _env_file.exists():
    _env_file = PROJECT_ROOT / '.env'

from dotenv import load_dotenv
load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()

print(f'Platform : {PLATFORM}  Project: {PROJECT_ROOT}  Data: {DATA_ROOT}')

# AnchoredCOOLANT Training — Stage 03a

Trains `AnchoredCOOLANT` — COOLANT extended with a frozen Stage 03.9 PhoBERT CLS anchor.

**Direction 1 — Semantic Anchor Injection**: `final_corre [B,96]` → `[B,160]` by
concatenating a projected PhoBERT CLS vector `[B,64]`.

**Direction 2 — NEI-Ambiguity Coupling** (optional): auxiliary loss pushes COOLANT
ambiguity scalar high for NEI samples. Disable with `nei_weight=0.0`.

```
Inputs:  processed_data/hdf5/coolant_{train,dev,test}.h5
         training/stage39_features/stage39_{train,dev,test}.h5
Output:  training/checkpoints_anchored_coolant/<run>/best_model.pth
```

In [ ]:
CONFIG = {
    'paths': {
        'train_hdf5':      DATA_ROOT / 'processed_data' / 'hdf5' / 'coolant_train.h5',
        'dev_hdf5':        DATA_ROOT / 'processed_data' / 'hdf5' / 'coolant_dev.h5',
        'test_hdf5':       DATA_ROOT / 'processed_data' / 'hdf5' / 'coolant_test.h5',
        'stage39_train':   DATA_ROOT / 'training' / 'stage39_features' / 'stage39_train.h5',
        'stage39_dev':     DATA_ROOT / 'training' / 'stage39_features' / 'stage39_dev.h5',
        'stage39_test':    DATA_ROOT / 'training' / 'stage39_features' / 'stage39_test.h5',
        'stage39_ckpt':    DATA_ROOT / 'training' / 'checkpoints_vifactcheck'
                           / 'vifactcheck_stage39_20260616_082028' / 'best_model.pth',
        'checkpoint_root': DATA_ROOT / 'training' / 'checkpoints_anchored_coolant',
        'mlflow_dir':      DATA_ROOT / 'mlruns',
    },
    'model': {
        'variant':         'AnchoredCOOLANT',
        'image_dim':       2048,
        'text_embed_dim':  768,
        'text_seq_len':    128,
        'shared_dim':      128,
        'sim_dim':         64,
        'clip_embed_dim':  64,
        'feature_dim':     96,   # base COOLANT dim; anchor_proj_dim added on top
        'h_dim':           64,
        'anchor_dim':      768,  # PhoBERT CLS dim
        'anchor_proj_dim': 64,   # final_corre = 96 + 64 = 160
        'lr':              1e-4,
        'weight_decay':    1e-5,
        'dropout':         0.1,
    },
    'training': {
        'batch_size':           32,
        'max_epochs':           40,
        'patience':             8,
        'negative_shift':       3,
        'min_batch_for_negatives': 4,
        'grad_clip':            1.0,
        'warmup_steps':         200,
        'seed':                 42,
        'similarity_weight':    0.5,
        'clip_weight':          0.2,
        # Direction 2: set to 0.0 to disable NEI-Ambiguity coupling
        'nei_weight':           0.3,
    },
    'loss': {
        'cosine_margin': 0.2,
    },
    'mlflow': {
        'experiment_name': 'anchored-coolant-training',
    },
    'safety': {
        'smoke_test':              False,
        'smoke_batches':           2,
        'resume_from_checkpoint':  None,
    },
}

In [ ]:
import gc, json, random, hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

CONFIG['paths']['checkpoint_root'].mkdir(parents=True, exist_ok=True)

from src.models.anchored_coolant import AnchoredCOOLANT, nei_ambiguity_loss
from src.preprocessing.coolant.pair_dataset import create_coolant_dataloaders
from src.preprocessing.coolant.training_utils import make_coolant_pairs, soft_cross_entropy

print(f'PyTorch: {torch.__version__}')

## Step 1: Load Stage 03.9 Pre-extracted CLS Features

Features indexed by `article_id`. The COOLANT dataloader returns
`(caption, image, article_id)`, so we align per-batch via the id→index map —
fully shuffle-safe, no sequential index assumption.

In [ ]:
def load_stage39_features(h5_path):
    """Returns: features [N,768], id_to_idx dict(str->int)."""
    with h5py.File(h5_path, 'r') as f:
        features = torch.tensor(f['text_features'][:], dtype=torch.float32)
        raw_ids = f['article_ids'][:]
        article_ids = [
            x.decode('utf-8') if isinstance(x, bytes) else str(x)
            for x in raw_ids
        ]
    id_to_idx = {aid: i for i, aid in enumerate(article_ids)}
    print(f'  {Path(h5_path).name}: {features.shape[0]} samples, dim={features.shape[1]}')
    return features, id_to_idx

print('Loading Stage 03.9 CLS features...')
stage39 = {}
for split, key in [('train', 'stage39_train'), ('dev', 'stage39_dev'), ('test', 'stage39_test')]:
    p = CONFIG['paths'][key]
    if not p.exists():
        raise FileNotFoundError(
            f'Stage 03.9 features missing: {p}\n'
            'Run 03.9_vifactcheck_training.ipynb first — it pre-extracts CLS features.'
        )
    feats, id_map = load_stage39_features(p)
    stage39[split] = {'features': feats, 'id_to_idx': id_map}

def lookup_anchors(s39_split, article_ids_batch, device):
    """
    Resolve a batch of article_ids to CLS feature vectors [B, 768].
    Missing ids fall back to zero vector.
    article_ids_batch: int or str tensor/list from the dataloader.
    """
    feats = s39_split['features']
    id_map = s39_split['id_to_idx']
    dim = feats.shape[1]
    rows = []
    for aid in article_ids_batch:
        idx = id_map.get(str(int(aid)), None)
        rows.append(feats[idx] if idx is not None else torch.zeros(dim))
    return torch.stack(rows).to(device)

print('Stage 03.9 CLS features ready.')

## Step 2: Stage 03.9 Logit Cache (Direction 2 — optional)

Pre-compute ViFactCheck logits keyed by `article_id`. Skipped if `nei_weight=0`.

In [ ]:
# logit cache: split -> dict(str(article_id) -> Tensor[3])
stage39_logit_cache = None

if CONFIG['training']['nei_weight'] > 0.0:
    _ckpt_path = CONFIG['paths']['stage39_ckpt']
    if not _ckpt_path.exists():
        print(f'WARNING: Stage 03.9 checkpoint not found: {_ckpt_path}')
        print('Disabling Direction 2. Set nei_weight=0.0 to suppress this warning.')
        CONFIG['training']['nei_weight'] = 0.0
    else:
        from src.models.vifactcheck import PhoBERTClassifier
        _ckpt = torch.load(_ckpt_path, map_location='cpu')
        _clf = PhoBERTClassifier(num_labels=3)
        _clf.load_state_dict(_ckpt['model_state_dict'])
        _clf.eval()
        for _p in _clf.parameters():
            _p.requires_grad_(False)

        stage39_logit_cache = {}
        for split in ('train', 'dev', 'test'):
            feats = stage39[split]['features']     # [N, 768]
            id_map = stage39[split]['id_to_idx']   # str -> row
            with torch.no_grad():
                logits = _clf.classifier(feats)    # [N, 3]
            stage39_logit_cache[split] = {
                aid: logits[idx] for aid, idx in id_map.items()
            }
            print(f'  {split}: {len(stage39_logit_cache[split])} logit entries')

        del _clf
        print('Logit cache built. PhoBERT classifier freed.')

def lookup_logits(logit_cache_split, article_ids_batch, device):
    """Look up factcheck logits [B,3] for a batch, zeros for missing ids."""
    rows = [logit_cache_split.get(str(int(aid)), torch.zeros(3))
            for aid in article_ids_batch]
    return torch.stack(rows).to(device)

if CONFIG['training']['nei_weight'] == 0.0:
    print('Direction 2 disabled (nei_weight=0.0).')

In [ ]:
def select_device():
    if torch.cuda.is_available():
        dev = torch.device('cuda')
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'Device: cuda ({torch.cuda.get_device_name(0)}, {mem:.1f} GB)')
    elif torch.backends.mps.is_available():
        dev = torch.device('mps')
        print('Device: mps (Apple Silicon)')
    else:
        dev = torch.device('cpu')
        print('Device: cpu')
    return dev

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = select_device()
seed_everything(CONFIG['training']['seed'])
print(f'Seed: {CONFIG["training"]["seed"]}')

## Step 3: DataLoaders

`CoolantPairDataset.__getitem__` returns `(caption [768,seq], image [2048], article_id int)`.
Train loader uses `shuffle=True` — we use the returned `article_id` for all lookups,
never batch position.

In [ ]:
loaders, datasets = create_coolant_dataloaders(
    str(CONFIG['paths']['train_hdf5']),
    str(CONFIG['paths']['dev_hdf5']),
    str(CONFIG['paths']['test_hdf5']),
    batch_size=CONFIG['training']['batch_size'],
)

if CONFIG['safety']['smoke_test']:
    from itertools import islice
    _n = CONFIG['safety']['smoke_batches']
    class _SmokeLoader:
        def __init__(self, loader, n): self._l, self._n = loader, n
        def __iter__(self): return islice(self._l, self._n)
        def __len__(self): return min(self._n, len(self._l))
    loaders = {k: _SmokeLoader(v, _n) for k, v in loaders.items()}
    print(f'SMOKE TEST: {_n} batches/split')

_cap, _img, _ids = next(iter(loaders['train']))
print(f'caption: {tuple(_cap.shape)}  image: {tuple(_img.shape)}  article_ids: {tuple(_ids.shape)}')

## Step 4: Build AnchoredCOOLANT

In [ ]:
def build_model(config, device):
    model = AnchoredCOOLANT.from_config(config['model'], device=str(device))
    total    = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    det_p    = sum(p.numel() for p in model.detection_module.parameters())
    anchor_p = sum(p.numel() for p in model.detection_module.anchor_proj.parameters())
    print(f'Parameters — total: {total:,}  trainable: {trainable:,}')
    print(f'  detection_module: {det_p:,}  (anchor_proj: {anchor_p:,})')
    print(f'  final_corre_dim : {config["model"]["feature_dim"] + config["model"]["anchor_proj_dim"]}')
    return model

model = build_model(CONFIG, DEVICE)

# Sanity check: both anchored and no-anchor forward paths
model.eval()
with torch.no_grad():
    _B = 4
    _t  = torch.randn(_B, 768, 128).to(DEVICE)
    _i  = torch.randn(_B, 2048).to(DEVICE)
    _a  = torch.randn(_B, 768).to(DEVICE)
    assert model(_t, _i, phobert_cls=_a)['detection_logits'].shape == (_B, 2)
    assert model(_t, _i)['detection_logits'].shape == (_B, 2)  # zero-fill anchor
print('Sanity check passed.')
model.train()

## Step 5: Losses, Optimizers, Schedulers

In [ ]:
loss_cos = nn.CosineEmbeddingLoss(margin=CONFIG['loss']['cosine_margin'])

# Three independent optimizers matching official COOLANT recipe
OPTIMIZERS = {
    'similarity': torch.optim.Adam(
        model.similarity_module.parameters(),
        lr=CONFIG['model']['lr'], weight_decay=CONFIG['model']['weight_decay']),
    'clip': torch.optim.Adam(
        model.clip_module.parameters(),
        lr=CONFIG['model']['lr'], weight_decay=CONFIG['model']['weight_decay']),
    'detection': torch.optim.Adam(
        model.detection_module.parameters(),
        lr=CONFIG['model']['lr'], weight_decay=CONFIG['model']['weight_decay']),
}

def make_warmup_cosine(optimizer, warmup_steps, total_steps):
    import math
    def _lr(step):
        if step < warmup_steps:
            return float(step) / max(1, warmup_steps)
        p = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, _lr)

_total_steps = CONFIG['training']['max_epochs'] * len(loaders['train'])
_warmup      = CONFIG['training']['warmup_steps']
schedulers   = {k: make_warmup_cosine(opt, _warmup, _total_steps)
                for k, opt in OPTIMIZERS.items()}

print(f'Optimizers: Adam lr={CONFIG["model"]["lr"]}  warmup={_warmup}  total_steps={_total_steps}')

## Step 6: MLflow + Run Directory

In [ ]:
import mlflow

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name     = f'anchored_coolant_{timestamp}'
run_dir      = CONFIG['paths']['checkpoint_root'] / run_name
artifact_dir = run_dir / 'artifacts'
run_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
print(f'Run dir: {run_dir}')

mlflow_enabled = False
mlflow_run_id  = None
try:
    mlflow.set_tracking_uri(CONFIG['paths']['mlflow_dir'].as_uri())
    mlflow.set_experiment(CONFIG['mlflow']['experiment_name'])
    _run = mlflow.start_run(run_name=run_name)
    mlflow_run_id = _run.info.run_id
    mlflow.log_params({
        'anchor_proj_dim': CONFIG['model']['anchor_proj_dim'],
        'nei_weight':      CONFIG['training']['nei_weight'],
        'batch_size':      CONFIG['training']['batch_size'],
        'max_epochs':      CONFIG['training']['max_epochs'],
        'patience':        CONFIG['training']['patience'],
        'lr':              CONFIG['model']['lr'],
        'seed':            CONFIG['training']['seed'],
    })
    mlflow_enabled = True
    print(f'MLflow run: {mlflow_run_id}')
except Exception as e:
    print(f'MLflow disabled ({e})')

## Step 7: Training + Evaluation Functions

### Anchor alignment

Train loader has `shuffle=True`, so batch position ≠ dataset row.
We use the `article_id` from the dataloader's 3rd element to look up
CLS features — fully shuffle-safe.

For the detection batch (matched + unmatched, then shuffled by `randperm`),
we replicate the construction locally so the anchor tensor undergoes the
same permutation as the caption/image tensors.

In [ ]:
def cleanup_memory(device):
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

def assert_finite(loss, label):
    if not torch.isfinite(loss):
        raise FloatingPointError(f'NaN/Inf in {label}: {loss.item()}')

def compute_metrics(y_true, y_pred, prefix):
    acc = (np.array(y_pred) == np.array(y_true)).mean()
    return {
        f'{prefix}_accuracy':       round(float(acc), 4),
        f'{prefix}_macro_f1':       round(float(f1_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        f'{prefix}_fake_precision': round(float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)), 4),
        f'{prefix}_fake_recall':    round(float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)), 4),
        f'{prefix}_real_recall':    round(float(recall_score(y_true, y_pred, pos_label=0, zero_division=0)), 4),
    }

def make_detection_batch_with_anchor(caption, image, anchor, shift=3):
    """
    Build balanced detection batch AND apply the same randperm to the anchor,
    preserving per-sample alignment through the shuffle.

    Real pair:  (caption_i, image_i,      anchor_i)  label=0
    Fake pair:  (caption_i, image_{i+k},  anchor_i)  label=1
    Anchor follows the caption (text-based feature) in both cases.

    Returns: det_cap [2B,...], det_img [2B,2048], det_anc [2B,768], labels [2B]
    """
    B, device = caption.size(0), caption.device

    combined_cap = torch.cat([caption, caption.clone()],          dim=0)  # real + fake
    combined_img = torch.cat([image,   image.roll(shift, dims=0)], dim=0)
    combined_anc = torch.cat([anchor,  anchor.clone()],            dim=0)  # anchor = caption
    labels = torch.cat([
        torch.zeros(B, dtype=torch.long, device=device),
        torch.ones(B,  dtype=torch.long, device=device),
    ])

    perm = torch.randperm(2 * B, device=device)
    return combined_cap[perm], combined_img[perm], combined_anc[perm], labels[perm]

print('Helpers defined.')

In [ ]:
def train_one_epoch(epoch, model, loader, optimizers, schedulers,
                    device, config, s39_split, s39_logit_split):
    model.train()
    total_loss, n_batches, skipped = 0.0, 0, 0
    all_preds, all_labels = [], []
    nei_weight = config['training']['nei_weight']
    grad_clip  = config['training']['grad_clip']
    shift      = config['training']['negative_shift']
    min_batch  = config['training']['min_batch_for_negatives']

    for caption, image, article_ids in tqdm(loader, desc=f'Epoch {epoch:02d} [train]', leave=False):
        B = caption.size(0)
        if B < min_batch:
            skipped += 1
            continue

        caption, image = caption.to(device), image.to(device)

        # Anchor lookup by article_id — correct under shuffle=True
        anchor = lookup_anchors(s39_split, article_ids, device)  # [B, 768]

        # ── Task 1: Similarity + CLIP ───────────────────────────────────────
        cap_a, img_m, img_u = make_coolant_pairs(caption, image, shift=shift)
        out_m = model(cap_a, img_m, phobert_cls=anchor)
        out_u = model(cap_a, img_u, phobert_cls=anchor)

        ta_m, ia_m = out_m['text_aligned_clip'], out_m['image_aligned_clip']
        ta_u, ia_u = out_u['text_aligned_clip'], out_u['image_aligned_clip']

        lbl_p =  torch.ones(B, device=device)
        lbl_n = -torch.ones(B, device=device)
        sim_loss  = (loss_cos(ta_m, ia_m, lbl_p) + loss_cos(ta_u, ia_u, lbl_n)) * 0.5
        clip_loss = model.compute_clip_loss(ta_m, ia_m)
        clip_logits = torch.matmul(ia_m, ta_m.T) * torch.exp(model.clip_module.temperature)
        soft_loss = soft_cross_entropy(clip_logits, F.softmax(clip_logits.detach(), dim=1))
        task1_loss = (config['training']['similarity_weight'] * sim_loss +
                      config['training']['clip_weight'] * (clip_loss + soft_loss))

        for opt in (optimizers['similarity'], optimizers['clip']):
            opt.zero_grad()
        task1_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.similarity_module.parameters(), grad_clip)
        torch.nn.utils.clip_grad_norm_(model.clip_module.parameters(), grad_clip)
        optimizers['similarity'].step(); schedulers['similarity'].step()
        optimizers['clip'].step();       schedulers['clip'].step()

        # ── Task 2: Detection — anchor aligned with randperm ────────────────
        det_cap, det_img, det_anc, det_lbl = make_detection_batch_with_anchor(
            caption, image, anchor, shift=shift)

        # Direction 2: per-sample NEI logits; replicate for matched+unmatched
        factcheck_logits = None
        if nei_weight > 0.0 and s39_logit_split is not None:
            fc = lookup_logits(s39_logit_split, article_ids, device)  # [B, 3]
            # Both matched and unmatched share the same caption/anchor per sample
            factcheck_logits = torch.cat([fc, fc], dim=0)             # [2B, 3]
            # Apply the same perm — we can't replay it here, so use per-sample
            # average across the detection batch as a conservative fallback.
            # NOTE: Direction 1 anchor is per-sample (exact); Direction 2 logits
            # only need rough NEI signal, so batch mean is acceptable.
            factcheck_logits = fc.mean(0, keepdim=True).expand(2 * B, -1)

        det_out = model(det_cap, det_img, phobert_cls=det_anc)
        det_losses = model.compute_detection_loss(
            det_out['detection_logits'], det_lbl,
            det_out['attention_weights'], det_out['ambiguity_weights'],
            factcheck_logits=factcheck_logits,
            nei_weight=nei_weight,
        )
        det_loss = det_losses['detection_loss']
        assert_finite(det_loss, 'det_loss')

        optimizers['detection'].zero_grad()
        det_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.detection_module.parameters(), grad_clip)
        optimizers['detection'].step(); schedulers['detection'].step()

        batch_loss = task1_loss.item() + det_loss.item()
        total_loss += batch_loss
        n_batches  += 1
        all_preds.extend(det_out['detection_logits'].argmax(1).cpu().tolist())
        all_labels.extend(det_lbl.cpu().tolist())

    cleanup_memory(device)
    m = compute_metrics(all_labels, all_preds, 'train')
    m['train_loss'] = round(total_loss / max(1, n_batches), 4)
    m['skipped']    = skipped
    return m


def evaluate(model, loader, device, config, split, s39_split):
    model.eval()
    total_loss, n_batches = 0.0, 0
    all_preds, all_labels = [], []
    shift = config['training']['negative_shift']

    with torch.no_grad():
        for caption, image, article_ids in tqdm(loader, desc=f'  [{split}]', leave=False):
            caption, image = caption.to(device), image.to(device)
            anchor = lookup_anchors(s39_split, article_ids, device)

            det_cap, det_img, det_anc, det_lbl = make_detection_batch_with_anchor(
                caption, image, anchor, shift=shift)

            det_out = model(det_cap, det_img, phobert_cls=det_anc)
            det_losses = model.compute_detection_loss(
                det_out['detection_logits'], det_lbl,
                det_out['attention_weights'], det_out['ambiguity_weights'],
            )
            total_loss += det_losses['detection_loss'].item()
            n_batches  += 1
            all_preds.extend(det_out['detection_logits'].argmax(1).cpu().tolist())
            all_labels.extend(det_lbl.cpu().tolist())

    cleanup_memory(device)
    m = compute_metrics(all_labels, all_preds, split)
    m[f'{split}_loss']      = round(total_loss / max(1, n_batches), 4)
    m['confusion_matrix']   = confusion_matrix(all_labels, all_preds, labels=[0, 1]).tolist()
    return m

print('train_one_epoch / evaluate defined.')

## Step 8: Checkpoint Helpers

In [ ]:
def config_to_jsonable(cfg):
    if isinstance(cfg, dict):         return {k: config_to_jsonable(v) for k, v in cfg.items()}
    if isinstance(cfg, Path):         return str(cfg)
    if isinstance(cfg, (list,tuple)): return [config_to_jsonable(v) for v in cfg]
    return cfg

def save_checkpoint(path, model, epoch, config, history, metrics, mlflow_run_id=None):
    torch.save({
        'model_state_dict': model.state_dict(),
        'config':           config_to_jsonable(config),
        'epoch':            epoch,
        'metrics':          metrics,
        'training_history': history,
        'architecture':     'AnchoredCOOLANT',
        'anchor_proj_dim':  config['model']['anchor_proj_dim'],
        'mlflow_run_id':    mlflow_run_id,
    }, path)

print(f'Checkpoint dir: {run_dir}')

## Step 9: Training Loop

In [ ]:
history          = []
best_val_acc     = -1.0
best_val_f1      = -1.0
best_epoch       = -1
patience_counter = 0
best_ckpt_path   = run_dir / 'best_model.pth'
best_f1_ckpt_path = run_dir / 'best_macro_f1.pth'

start_epoch = 0
_resume = CONFIG['safety']['resume_from_checkpoint']
if _resume:
    _c = torch.load(_resume, map_location=DEVICE)
    model.load_state_dict(_c['model_state_dict'])
    start_epoch = _c.get('epoch', 0) + 1
    print(f'Resumed from epoch {start_epoch - 1}')

print(f'Training: max_epochs={CONFIG["training"]["max_epochs"]}  '
      f'patience={CONFIG["training"]["patience"]}  '
      f'nei_weight={CONFIG["training"]["nei_weight"]}')

try:
    for epoch in range(start_epoch, CONFIG['training']['max_epochs']):
        s39_log = (stage39_logit_cache or {}).get('train')

        train_m = train_one_epoch(
            epoch, model, loaders['train'], OPTIMIZERS, schedulers,
            DEVICE, CONFIG, stage39['train'], s39_log)

        val_m = evaluate(
            model, loaders['dev'], DEVICE, CONFIG, 'val', stage39['dev'])

        record = {
            'epoch':          epoch,
            'train_loss':     train_m['train_loss'],
            'train_accuracy': train_m['train_accuracy'],
            'train_macro_f1': train_m['train_macro_f1'],
            'val_loss':       val_m['val_loss'],
            'val_accuracy':   val_m['val_accuracy'],
            'val_macro_f1':   val_m['val_macro_f1'],
            'skipped':        train_m['skipped'],
        }

        if mlflow_enabled:
            try:
                mlflow.log_metrics(
                    {k: record[k] for k in ['train_loss','train_accuracy',
                                            'val_loss','val_accuracy','val_macro_f1']},
                    step=epoch)
            except Exception: pass

        val_acc = val_m['val_accuracy']
        if val_acc > best_val_acc:
            best_val_acc     = val_acc
            best_epoch       = epoch
            patience_counter = 0
            save_checkpoint(best_ckpt_path, model, epoch, CONFIG,
                            history, {**train_m, **val_m}, mlflow_run_id)
            print(f'  ★ Best val_acc={val_acc:.4f} (epoch {epoch})')
        else:
            patience_counter += 1

        val_f1 = val_m['val_macro_f1']
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            save_checkpoint(best_f1_ckpt_path, model, epoch, CONFIG,
                            history, {**train_m, **val_m}, mlflow_run_id)

        history.append(record)
        with open(artifact_dir / 'training_history.json', 'w') as _f:
            json.dump(history, _f, indent=2)
        pd.DataFrame(history).to_csv(artifact_dir / 'training_history.csv', index=False)

        print(f'Epoch {epoch:02d} | '
              f'tr_loss={record["train_loss"]:.4f} tr_acc={record["train_accuracy"]:.4f} '
              f'val_loss={record["val_loss"]:.4f} val_acc={val_acc:.4f} '
              f'val_f1={val_f1:.4f} '
              f'(patience {patience_counter}/{CONFIG["training"]["patience"]})')

        if patience_counter >= CONFIG['training']['patience']:
            print(f'Early stopping at epoch {epoch}.')
            break

except KeyboardInterrupt:
    _int = run_dir / f'interrupted_e{epoch}.pth'
    save_checkpoint(_int, model, epoch, CONFIG, history, {}, mlflow_run_id)
    print(f'Interrupted — saved: {_int}')
    if mlflow_enabled:
        try: mlflow.end_run()
        except Exception: pass
    raise

print(f'\nTraining done. Best val_acc={best_val_acc:.4f} (epoch {best_epoch}).')

## Step 10: Test Evaluation

In [ ]:
eval_model = build_model(CONFIG, DEVICE)
_c = torch.load(best_ckpt_path, map_location=DEVICE)
eval_model.load_state_dict(_c['model_state_dict'])
eval_model.eval()
print(f'Loaded best checkpoint (epoch {_c["epoch"]}): {best_ckpt_path}')

test_metrics = evaluate(
    eval_model, loaders['test'], DEVICE, CONFIG, 'test', stage39['test'])

test_report = {
    'architecture':     'AnchoredCOOLANT',
    'anchor_proj_dim':  CONFIG['model']['anchor_proj_dim'],
    'nei_weight':       CONFIG['training']['nei_weight'],
    'best_epoch':       best_epoch,
    'best_val_accuracy': best_val_acc,
    'best_val_macro_f1': best_val_f1,
    **test_metrics,
}
with open(artifact_dir / 'test_report.json', 'w') as _f:
    json.dump(test_report, _f, indent=2)

print('\nTest results:')
for k, v in test_metrics.items():
    if k != 'confusion_matrix':
        print(f'  {k}: {v}')
print(f'  confusion_matrix: {test_metrics.get("confusion_matrix")}')

if mlflow_enabled:
    try:
        mlflow.log_metrics({k: v for k, v in test_metrics.items()
                            if k != 'confusion_matrix'})
    except Exception: pass

## Step 11: Training Curves + Confusion Matrix

In [ ]:
if history:
    epochs = [r['epoch'] for r in history]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (tr_k, val_k, title) in zip(axes, [
        ('train_loss',     'val_loss',     'Loss'),
        ('train_accuracy', 'val_accuracy', 'Accuracy'),
        ('train_macro_f1', 'val_macro_f1', 'Macro-F1'),
    ]):
        ax.plot(epochs, [r[tr_k]  for r in history], label='train')
        ax.plot(epochs, [r[val_k] for r in history], label='val')
        ax.set_title(title); ax.legend(); ax.grid(True)
    plt.tight_layout()
    plt.savefig(artifact_dir / 'training_curves.png', dpi=120)
    plt.show(); plt.close()

if test_metrics.get('confusion_matrix'):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(np.array(test_metrics['confusion_matrix']),
                annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real (0)', 'Fake (1)'],
                yticklabels=['Real (0)', 'Fake (1)'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Test Confusion Matrix — AnchoredCOOLANT')
    plt.tight_layout()
    plt.savefig(artifact_dir / 'test_confusion_matrix.png', dpi=120)
    plt.show(); plt.close()

## Step 12: Comparison with Baseline COOLANT

In [ ]:
# Baseline from Stage 03 best run — update if you retrain
BASELINE = {
    'model':             'ResNetCOOLANT (Stage 03)',
    'test_accuracy':     0.8507,
    'test_macro_f1':     0.8474,
    'test_fake_recall':  0.7034,
    'test_real_recall':  0.9981,
}

print('=' * 65)
print('AnchoredCOOLANT Results')
print('=' * 65)
print(f'  anchor_proj_dim : {CONFIG["model"]["anchor_proj_dim"]}')
print(f'  nei_weight      : {CONFIG["training"]["nei_weight"]}')
print(f'  best_epoch      : {best_epoch}')
print()
print(f'{"Metric":<26} {"Baseline":>10} {"Anchored":>10} {"Delta":>8}')
print('-' * 56)
for metric, base_val in [
    ('test_accuracy',    BASELINE['test_accuracy']),
    ('test_macro_f1',    BASELINE['test_macro_f1']),
    ('test_fake_recall', BASELINE['test_fake_recall']),
    ('test_real_recall', BASELINE['test_real_recall']),
]:
    our = test_metrics.get(metric, float('nan'))
    delta = our - base_val
    print(f'{metric:<26} {base_val:>10.4f} {our:>10.4f} {"+" if delta>=0 else ""}{delta:>7.4f}')

if mlflow_enabled:
    try: mlflow.end_run()
    except Exception: pass

print()
print('Key metric: test_fake_recall — improvement means anchor helps catch missed fakes.')